# Power traces in PowerTrace-Sim's format, on their 250 ms grid

One thing only: the serving-engine power trace, sampled and drawn the way
PowerTrace-Sim does it, so a figure from here can be put beside one of theirs.

No dashboards, no batch-composition panels, no KV plots, no work vectors. Those live
in `Serving_Engine_Power_Sim_Colab.ipynb`.

---

## Where 250 ms comes from

**It is not an analysis choice.** It is the rate their profiling jobs poll the sensor
at. From `profiling/jobs/llama-3-8b.sh`, and every other job script in that directory:

```bash
nvidia-smi --query-gpu=timestamp,power.draw,utilization.gpu,memory.used \
           --format=csv -lms 250 >> ${OUTPUT_PREFIX}.csv &
```

`-lms 250` is a 250 ms loop -- 4 Hz. So 250 ms is the **finest resolution at which any
comparison with one of their captures means anything.** A simulator can emit power at
kernel resolution, but no meter in that campaign ever recorded it, and a figure drawn
finer than the sensor shows detail that could never be checked against anything.

That constraint propagates through their code: `model/release.py` refuses an artifact
whose `native_dt_s != 0.25`, and `evaluation_core.trace_metrics` defaults to
`native_dt=0.25`.

## And the rest of their protocol

| | their value | source |
|---|---|---|
| sensor rate | **250 ms** | `nvidia-smi -lms 250` |
| run length | **600 s** per rate | `NUM_PROMPTS = 600 * ARRIVAL_RATE` |
| crop | **600 s** | `TRACE_SECONDS = 600.0` |
| plotted at | **1 s means** | `one_second_pair`, four native samples |
| rate ladder | powers of 4 | `ARRIVAL_RATES=(0.0625 0.25 1 4 16 64)` |
| y-axis | **per GPU** | divided by `tp` |

Everything here follows that, with one deliberate departure flagged in section 3.

## 1 - Install

In [ ]:
import os, subprocess, sys

REPO = 'https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git'
DIR  = '/content/dynamic_shape_power_sim'

if not os.path.isdir(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, DIR], check=True)
else:
    subprocess.run(['git', '-C', DIR, 'pull', '--ff-only'], check=True)

if DIR not in sys.path:
    sys.path.insert(0, DIR)
os.chdir(DIR)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy', 'pandas', 'matplotlib', 'seaborn'], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import dynshape
from dynshape import (ShapeRewriter, build_predictor, TrafficConfig,
                      generate_traffic, EngineConfig, SchedulerConfig,
                      run_engine, reset_ids)
from dynshape.engine_plot import (FSTS_NATIVE_DT_S, FSTS_PLOT_DT_S,
                                  fsts_grids, plot_power_trace_paper,
                                  trace_agreement)

pd.set_option('display.width', 170)
print('dynshape', dynshape.__version__)
print('native grid', FSTS_NATIVE_DT_S, 's   plotted at', FSTS_PLOT_DT_S, 's')

## 2 - The engine

Unchanged from the other notebooks. The predictor is the roofline unless the
EnergAIzer LUT is present, and every label says which.

In [ ]:
rw   = ShapeRewriter.from_dir('templates/gpt2')
pred = build_predictor(force_analytic=True)

IS_MEASURED = pred.backend.is_measured_model
SOURCE = 'EnergAIzer LUT' if IS_MEASURED else 'Roofline (SYNTHETIC)'
print('predictor:', pred.backend.name)

## 3 - The one departure, and why

FSTS runs **600 s per rate** and their ladder tops out at 64 req/s. Both numbers are
calibrated to the models they profile -- Llama-3-8B through gpt-oss-120b, where a
request takes on the order of a second.

GPT-2 is roughly **two orders of magnitude faster**: a 280-token prefill is about 1 ms
and a decode step about 0.8 ms. Two consequences, pulling in opposite directions:

- **their rates would leave the GPU idle.** At 1 req/s a GPT-2 replica has 0.03
  requests in flight. The trace would be a flat 47 W floor with occasional
  single-request spikes -- a correct picture of an idle machine, and a useless one.
- **600 s at a rate that does load it is a very long simulation.** At 100 req/s the
  engine runs ~220 iterations per simulated second, so 600 s is ~130,000 iterations of
  roughly 1,400 kernels each. That is not a Colab cell.

So: **their grid, their figure, their metrics -- a shorter window and a ladder scaled
to the model.** Set `WINDOW_S = 600` to reproduce their crop exactly if you have the
time; the default is sized to finish while you watch.

The ladder keeps their *shape* -- successive doublings spanning idle to loaded --
rather than their absolute values, which describe a different model's capacity.

In [ ]:
WINDOW_S = 20.0                     # FSTS: 600.0
RATES    = [15, 30, 60, 120]        # FSTS: 0.0625 0.25 1 4 16 64, for slower models
SEED     = 7

print(f'{WINDOW_S:.0f} s at {FSTS_NATIVE_DT_S*1000:.0f} ms '
      f'= {int(WINDOW_S/FSTS_NATIVE_DT_S)} native samples per rate')
print(f'{WINDOW_S:.0f} s at {FSTS_PLOT_DT_S:.0f} s '
      f'= {int(WINDOW_S/FSTS_PLOT_DT_S)} one-second means per rate')
print()
print('theirs:  600 s -> 2400 native, 600 plotted')
print()
print('Expect a few minutes. The LOW rates are not the cheap ones: with nothing to')
print('batch, the engine runs one iteration per decode token, so 15 req/s costs')
print('nearly as much to simulate as 120 does.')

In [ ]:
import time

traces = {}
for rate in RATES:
    reset_ids()
    # Their rule: NUM_PROMPTS = window * ARRIVAL_RATE.
    n = int(round(WINDOW_S * rate))
    reqs = generate_traffic(TrafficConfig(
        interval='poisson', qps=float(rate),
        length='zipf', min_tokens=64, max_tokens=1024, theta=0.85,
        prefill_to_decode_ratio=4.0, num_requests=n, seed=SEED))
    t0 = time.time()
    traces[rate] = run_engine(reqs, rw, pred, EngineConfig(
        scheduler=SchedulerConfig(chunk_size=2048, max_num_seqs=256,
                                  block_size=16, max_tokens=4096),
        record_kernels_until_ms=0.0))
    s = traces[rate].summary()
    print(f"rate {rate:>4} req/s   {n:>5} requests   "
          f"{s['wall_time_s']:6.1f} s simulated   "
          f"{s['iterations']:>6} iterations   "
          f"duty {s['duty_cycle']:.0%}   "
          f"[{time.time()-t0:.0f} s to run]")

## 4 - The figure

Their rate-ladder layout: one panel per rate, **shared y-axis**, at paper-column size.
The shared ceiling is the whole point -- four independently autoscaled panels would
make every trace look like the same shape.

Drawn on the **native 250 ms grid**. Their published figures use 1 s means over a 600 s
run, which is 600 points; a 20 s window at 1 s would be 20, so the native grid is what
keeps this legible at this window. Section 5 shows what their 1 s convention does to it.

The x-axis is seconds rather than minutes for the same reason: theirs spans ten
minutes, this spans twenty seconds, and labelling that `0.33 min` would be silly.

In [ ]:
ymax = 1.05 * max(traces[r].resample(dt_ms=FSTS_NATIVE_DT_S*1000)[1].max()
                  for r in RATES)

fig, axes = plt.subplots(1, len(RATES), figsize=(4.4*len(RATES), 2.5), sharey=True)
for ax, rate in zip(axes, RATES):
    plot_power_trace_paper([(f'{SOURCE}   {rate} req/s', traces[rate])],
                           dt_s=FSTS_NATIVE_DT_S, ax=ax, ylim=ymax)
for ax in axes[1:]:
    ax.set_ylabel('')
plt.tight_layout(pad=0.4, rect=(0, 0, 1, 0.86))
plt.show()

print(f'{int(WINDOW_S/FSTS_NATIVE_DT_S)} samples per panel, on the grid their sensor')
print('records at, with one shared ceiling across all four.')

## 5 - The native grid against the plotted one

They **sample** at 250 ms because that is what the sensor gives, and **plot** at 1 s
because four-sample means are what the figures and the published metrics are defined on.

Both are box filters on aligned bins, so averaging four 250 ms samples and resampling
once to 1 s give the identical answer -- a test pins that rather than assuming it. What
differs is the **peak**: a longer aperture cannot invent a spike, only average one away.
A peak quoted off the 250 ms series is not the number in their 1 s tables.

In [ ]:
rate = RATES[-1]
t_n, p_n, t_p, p_p = fsts_grids(traces[rate])

fig, ax = plt.subplots(figsize=(8.8, 2.7))
ax.plot(t_n/1000, p_n, lw=0.6, color='#999999',
        label=f'native {FSTS_NATIVE_DT_S*1000:.0f} ms  (what the sensor records)')
ax.plot(t_p/1000, p_p, lw=1.4, color='#8C1515',
        label=f'plotted {FSTS_PLOT_DT_S:.0f} s  (four-sample means)')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Power per GPU (W)')
ax.set_ylim(0, 1.05*p_n.max()); ax.set_xlim(0, t_n[-1]/1000)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), frameon=False,
          ncol=2, fontsize=8)
ax.grid(alpha=0.25)
plt.tight_layout(pad=0.4, rect=(0, 0, 1, 0.9)); plt.show()

usable = (p_n.size // 4) * 4
manual = p_n[:usable].reshape(-1, 4).mean(axis=1)
print(f'four-sample means vs a direct 1 s resample: '
      f'max difference {np.abs(manual - p_p[:manual.size]).max():.2e} W')
print()
print(f'peak at {FSTS_NATIVE_DT_S*1000:.0f} ms : {p_n.max():7.2f} W')
print(f'peak at {FSTS_PLOT_DT_S:.0f} s    : {p_p.max():7.2f} W'
      f'   ({100*(1 - p_p.max()/p_n.max()):.1f}% lower)')
print(f'mean, both grids  : {p_n.mean():7.2f} / {p_p.mean():7.2f} W  (invariant)')

## 6 - Their metrics across the ladder

From `feature-test/evaluation_core.py`, computed on the native 250 ms grid as they do,
with the lowest rate as the reference.

Read these as *how the trace changes with load*, not as an accuracy claim -- neither
series here is hardware.

In [ ]:
ref = traces[RATES[0]]
rows = []
for rate in RATES:
    m = trace_agreement(ref, traces[rate], dt_s=FSTS_NATIVE_DT_S)
    s = traces[rate].summary()
    rows.append({'rate (req/s)': rate,
                 'mean W': s['avg_power_w_wallclock'],
                 'duty cycle': s['duty_cycle'],
                 **{k: m[k] for k in ('mean_bias_pct', 'nrmse_range',
                                      'acf_r2', 'ks_agreement')}})
display(pd.DataFrame(rows).round(4))

print(f'Reference: {RATES[0]} req/s. Compared on the '
      f'{FSTS_NATIVE_DT_S*1000:.0f} ms grid, over the window the two runs share.')
print()
print('ks_agreement falling with load is the trace visiting different power LEVELS;')
print('acf_r2 falling is it wobbling on different TIMESCALES. A mean-power column')
print('alone reports neither, which is why they publish five numbers and not one.')

## 7 - Export, in their column format

Matching the CSV `plot_best_rate_traces.py` writes beside its figures, so the two can be
concatenated and plotted by the same script.

In [ ]:
frames = []
for rate in RATES:
    t_n, p_n, _, _ = fsts_grids(traces[rate])
    frames.append(pd.DataFrame({
        'rate_requests_s': rate,
        'run_id': f'gpt2_a100_tp1_rate_{rate}',
        'time_s': t_n / 1000.0,
        'power_w_per_gpu': p_n,
        'source': SOURCE,
        'native_dt_s': FSTS_NATIVE_DT_S}))
out = pd.concat(frames, ignore_index=True)
out.to_csv('power_trace_gpt2_a100_tp1_250ms.csv', index=False)
print('power_trace_gpt2_a100_tp1_250ms.csv', out.shape)
display(out.head())

try:
    from google.colab import files
    files.download('power_trace_gpt2_a100_tp1_250ms.csv')
except Exception:
    print('(not on Colab)')

## What this figure is, and is not

**Is:** a simulated GPT-2 power trace on the same time grid, in the same figure format,
with the same metrics as PowerTrace-Sim's published traces. Put side by side, the
rendering will not be what differs.

**Is not a measurement.** In their figures the black line is an `nvidia-smi` capture
from real hardware. Nothing here is that -- the best available reference is EnergAIzer's
measured LUT (`EnergAIzer_LUT_Power_Colab.ipynb`), and with the roofline predictor it is
not even that. The labels say which.

**Not a like-for-like comparison either**, for two reasons worth keeping in view: GPT-2
on one A100 is a far smaller system than Llama-3-8B on an H100, and the window and rate
ladder here are scaled to it rather than copied.